# Feature Engineering — Phiên bản tối ưu



In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define the base data path in Google Drive
DATA_PATH = '/content/drive/MyDrive/AIO2026_Prj/Datathon2026/datathon-2026-round-1'

# Create the data directory if it doesn't exist (within the specified DATA_PATH)
if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_PATH)
    print(f'Created directory: {DATA_PATH}')
else:
    print(f'Directory already exists: {DATA_PATH}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directory already exists: /content/drive/MyDrive/AIO2026_Prj/Datathon2026/datathon-2026-round-1


## 1. Constants

In [ ]:
# =========================================================
# Tết Nguyên Đán — ngày mồng 1 Tết Âm lịch
# =========================================================
TET_DATES = {
    2012: '2012-01-23',
    2013: '2013-02-10', 2014: '2014-01-31', 2015: '2015-02-19',
    2016: '2016-02-08', 2017: '2017-01-28', 2018: '2018-02-16',
    2019: '2019-02-05', 2020: '2020-01-25', 2021: '2021-02-12',
    2022: '2022-02-01', 2023: '2023-01-22', 2024: '2024-02-10',
    2025: '2025-01-29',
}
TET_TS = {y: pd.Timestamp(v) for y, v in TET_DATES.items()}

# =========================================================
# Lịch khuyến mãi — trích từ promotions.csv
# (name, start_month, start_day, duration_days, discount_pct, recurrence)
# recurrence: True = mọi năm | 'odd' = chỉ năm lẻ
# =========================================================
PROMO_SCHEDULE = [
    ('spring_sale',    3, 18, 30, 12,   True),
    ('mid_year',       6, 23, 29, 18,   True),
    ('fall_launch',    8, 30, 32, 10,   True),
    ('year_end',      11, 18, 45, 20,   True),
    ('urban_blowout',  7, 30, 33, None, 'odd'),   # chỉ năm lẻ — Q3 margin > 1.0
    ('rural_special',  1, 30, 30, 15,   'odd'),   # chỉ năm lẻ
]

# =========================================================
# Ngày lễ cố định Việt Nam
# =========================================================
VN_FIXED_HOLIDAYS = [
    (1,  1,  'new_year'),
    (3,  8,  'womens_day'),
    (4,  30, 'reunification'),
    (5,  1,  'labor_day'),
    (9,  2,  'national_day'),
    (10, 20, 'vn_womens_day'),
    (11, 11, 'dd_1111'),
    (12, 12, 'dd_1212'),
    (12, 24, 'christmas_eve'),
    (12, 25, 'christmas'),
]

print(f'Loaded {len(TET_DATES)} Tết dates, {len(PROMO_SCHEDULE)} promos, {len(VN_FIXED_HOLIDAYS)} holidays')

Loaded 14 Tết dates, 6 promos, 10 holidays


## 2. Helper functions

In [ ]:
def _nearest_tet_diff(dd):
    """
    Số ngày từ dd đến Tết gần nhất.
    Convention (theo AIO): dương = còn trước Tết, âm = đã qua Tết.
    Khác với teammate (dấu ngược lại).
    Giới hạn tìm kiếm trong ±60 ngày.
    """
    candidates = [
        TET_TS.get(dd.year - 1),
        TET_TS.get(dd.year),
        TET_TS.get(dd.year + 1),
    ]
    valid = [(c - dd).days for c in candidates if c is not None and abs((c - dd).days) <= 60]
    return min(valid, key=abs) if valid else 999


def _nearest_holiday_diff(dd, month, day):
    """
    Số ngày từ dd đến ngày lễ cố định gần nhất.
    Convention giống teammate: dương = đã qua, âm = chưa đến.
    """
    cands = [
        pd.Timestamp(year=dd.year - 1, month=month, day=day),
        pd.Timestamp(year=dd.year,     month=month, day=day),
        pd.Timestamp(year=dd.year + 1, month=month, day=day),
    ]
    diffs = [(dd - c).days for c in cands]
    abs_diffs = np.abs(diffs)
    return diffs[int(np.argmin(abs_diffs))]


def _is_black_friday(dd):
    if dd.month != 11:
        return 0
    last = pd.Timestamp(year=dd.year, month=11, day=30)
    last_fri = last - pd.Timedelta(days=(last.dayofweek - 4) % 7)
    return int(dd == last_fri)


print('Helper functions defined.')

Helper functions defined.


## 3. `build_features` — phiên bản kết hợp tối ưu

In [ ]:
def build_features(dates) -> pd.DataFrame:
    """
    Tạo toàn bộ feature từ DatetimeIndex / Series[datetime].
    Không dùng lịch sử Revenue/COGS → an toàn cho horizon 548 ngày.

    Tổng số features: ~100+
    """
    df   = pd.DataFrame({'Date': pd.to_datetime(dates)})
    date = df['Date']

    # =====================================================
    # (A) Calendar cơ bản
    # =====================================================
    df['year']          = date.dt.year
    df['month']         = date.dt.month
    df['day']           = date.dt.day
    df['dow']           = date.dt.dayofweek        # 0=Mon … 6=Sun
    df['doy']           = date.dt.dayofyear
    df['dim']           = date.dt.days_in_month    # days in month (cho Fourier monthly)
    df['quarter']       = date.dt.quarter
    df['is_weekend']    = (df['dow'] >= 5).astype(int)
    df['days_to_eom']   = df['dim'] - df['day']
    df['days_from_som'] = df['day'] - 1

    # =====================================================
    # (B) Edge-of-month — pattern MẠNH NHẤT trong data
    # Spike cuối tháng (ngày 28–31) và đầu tháng (ngày 1–3)
    # =====================================================
    for k in [1, 2, 3]:
        df[f'is_last{k}']  = (df['days_to_eom']  <= k - 1).astype(int)
        df[f'is_first{k}'] = (df['days_from_som'] <= k - 1).astype(int)

    # =====================================================
    # (C) Trend tuyến tính + Regime flags
    # 3 regime rõ ràng: pre-2019 / 2019 transition / post-2019
    # =====================================================
    df['t_days']          = (date - pd.Timestamp('2020-01-01')).dt.days
    df['t_years']         = df['t_days'] / 365.25
    df['regime_pre2019']  = (df['year'] <= 2018).astype(int)
    df['regime_2019']     = (df['year'] == 2019).astype(int)
    df['regime_post2019'] = (df['year'] >= 2020).astype(int)

    # =====================================================
    # (D) Fourier — mùa vụ đa tần số
    # FIX: teammate dùng range(1,5) → chỉ có bậc 1-4
    #      Đúng phải là range(1,6) để có bậc 1-5
    # =====================================================
    TAU = 2 * np.pi

    # Annual: bậc 1–5 (bắt mùa vụ năm)
    for i in range(1, 6):   # FIX: range(1,6) thay vì range(1,5)
        df[f'fourier_sin_y{i}'] = np.sin(TAU * i * (df['doy'] - 1) / 365.25)
        df[f'fourier_cos_y{i}'] = np.cos(TAU * i * (df['doy'] - 1) / 365.25)

    # Weekly: bậc 1–2 (bắt pattern tuần)
    # FIX: teammate chỉ có bậc 1, thêm bậc 2
    for i in range(1, 3):   # FIX: range(1,3) thay vì range(1,2)
        df[f'fourier_sin_w{i}'] = np.sin(TAU * i * df['dow'] / 7.0)
        df[f'fourier_cos_w{i}'] = np.cos(TAU * i * df['dow'] / 7.0)

    # Monthly: bậc 1–2 (bắt pattern trong tháng — EOM spike)
    # FIX: teammate chỉ có bậc 1, thêm bậc 2
    for i in range(1, 3):   # FIX: range(1,3) thay vì range(1,2)
        df[f'fourier_sin_m{i}'] = np.sin(TAU * i * (df['day'] - 1) / df['dim'])
        df[f'fourier_cos_m{i}'] = np.cos(TAU * i * (df['day'] - 1) / df['dim'])

    # =====================================================
    # (E) Holiday distance features — từ teammate
    # Mỗi ngày lễ tạo 4 features:
    #   - days_diff: khoảng cách có dấu (dương=đã qua, âm=chưa đến)
    #   - on_*: binary flag ngay đúng ngày
    #   - days_since_*: số ngày đã qua (trong 7 ngày sau lễ)
    #   - days_until_*: số ngày còn lại (trong 7 ngày trước lễ)
    # =====================================================
    for (month, day, holiday) in VN_FIXED_HOLIDAYS:
        hol_diffs = np.array([_nearest_holiday_diff(dd, month, day) for dd in date])
        df[f'hol_{holiday}_days_diff']   = hol_diffs
        df[f'on_{holiday}']              = (hol_diffs == 0).astype(int)
        # days_since: dương = đã qua lễ, window 7 ngày
        df[f'days_since_hol_{holiday}']  = np.where(
            (hol_diffs > 0) & (hol_diffs <= 7), hol_diffs, np.nan)
        # days_until: âm = chưa đến lễ, window 7 ngày
        df[f'days_until_hol_{holiday}']  = np.where(
            (hol_diffs < 0) & (hol_diffs >= -7), hol_diffs, np.nan)

    # =====================================================
    # (F) Tết Nguyên Đán — signal mạnh nhất trong năm
    # Convention: dương = còn trước Tết, âm = đã qua Tết
    # =====================================================
    tet_diffs = np.array([_nearest_tet_diff(dd) for dd in date])
    df['tet_days_diff']  = tet_diffs

    # Binary windows (AIO style — rõ ràng hơn)
    df['tet_in_7']        = (np.abs(tet_diffs) <= 7).astype(int)   # ±7 ngày quanh Tết
    df['tet_in_14']       = (np.abs(tet_diffs) <= 14).astype(int)  # ±14 ngày
    df['tet_before_7']    = ((tet_diffs > 0)  & (tet_diffs <= 7)).astype(int)   # 7 ngày trước Tết
    df['tet_before_14']   = ((tet_diffs > 0)  & (tet_diffs <= 14)).astype(int)  # 14 ngày trước Tết
    df['tet_after_7']     = ((tet_diffs < 0)  & (tet_diffs >= -7)).astype(int)  # 7 ngày sau Tết
    df['tet_on']          = (tet_diffs == 0).astype(int)            # đúng ngày mồng 1

    # Continuous windows (teammate style — cho model biết vị trí chính xác)
    df['days_since_tet']  = np.where(
        (tet_diffs < 0) & (tet_diffs >= -14), -tet_diffs, np.nan)  # 0-14 ngày sau Tết
    df['days_until_tet']  = np.where(
        (tet_diffs > 0) & (tet_diffs <= 14), tet_diffs, np.nan)    # 0-14 ngày trước Tết

    # THÊM MỚI: spike doanh thu 14-30 ngày sau Tết
    # Từ EDA AIO: doanh thu bứt lên mạnh ~4-4.5 triệu sau Tết 14-30 ngày
    df['tet_after_14_30'] = ((tet_diffs < -14) & (tet_diffs >= -30)).astype(int)

    # =====================================================
    # (G) Black Friday — mở rộng window ±3 ngày
    # Binary 1 ngày quá hẹp, shopping spree kéo dài cả tuần
    # =====================================================
    bf_diffs = []
    for dd in date:
        if dd.month != 11:
            bf_diffs.append(999)
            continue
        last = pd.Timestamp(year=dd.year, month=11, day=30)
        last_fri = last - pd.Timedelta(days=(last.dayofweek - 4) % 7)
        bf_diffs.append((dd - last_fri).days)
    bf_diffs = np.array(bf_diffs)

    df['on_black_friday']       = (bf_diffs == 0).astype(int)
    df['black_friday_window']   = (np.abs(bf_diffs) <= 3).astype(int)  # THÊM MỚI: ±3 ngày
    df['days_to_black_friday']  = np.where(
        (bf_diffs < 0) & (bf_diffs >= -7), bf_diffs, np.nan)           # THÊM MỚI: countdown

    # =====================================================
    # (H) Năm chẵn/lẻ + Interaction terms
    # Q3 năm lẻ: urban_blowout → margin > 1.0 (bán lỗ)
    # =====================================================
    df['is_odd_year'] = (df['year'] % 2).astype(int)

    # THÊM MỚI từ AIO: explicit interaction — model cây không tự tạo interaction
    df['q3_odd']      = ((df['quarter'] == 3) & (df['is_odd_year'] == 1)).astype(int)
    df['q3_even']     = ((df['quarter'] == 3) & (df['is_odd_year'] == 0)).astype(int)

    # =====================================================
    # (I) Promotion windows
    # FIX: range max(yrs)+2 thay vì +1 để cover năm 2024 trong test
    # =====================================================
    yrs = sorted(set(df['year'].tolist()))

    for (name, start_m, start_d, duration, discount_pct, recurrence) in PROMO_SCHEDULE:
        in_prom  = np.zeros(len(df), dtype=int)
        since    = np.full(len(df), -1.0)
        until    = np.full(len(df), -1.0)
        discount = np.zeros(len(df))

        # FIX: max(yrs)+2 thay vì max(yrs)+1
        for y in range(min(yrs) - 1, max(yrs) + 2):
            if recurrence == 'odd' and y % 2 == 0:
                continue
            start = pd.Timestamp(year=y, month=start_m, day=start_d)
            end   = start + pd.Timedelta(days=duration)
            mask  = (date >= start) & (date <= end)
            in_prom[mask]  = 1
            since[mask]    = (date[mask] - start).dt.days
            until[mask]    = (end - date[mask]).dt.days
            discount[mask] = discount_pct or 0

        df[f'promo_{name}']       = in_prom
        df[f'promo_{name}_since'] = since
        df[f'promo_{name}_until'] = until
        df[f'promo_{name}_disc']  = discount

    # THÊM MỚI: tổng hợp promo — signal cho stackable promotions
    promo_flag_cols = [f'promo_{name}' for name, *_ in PROMO_SCHEDULE]
    df['active_promo_count']  = df[promo_flag_cols].sum(axis=1)
    df['promo_any_active']    = (df['active_promo_count'] > 0).astype(int)
    df['promo_stackable']     = (df['active_promo_count'] > 1).astype(int)  # 2+ promo cùng lúc

    return df


# =====================================================
# Test build_features
# =====================================================
test_dates = pd.date_range('2023-01-01', '2023-01-10')
test_ft    = build_features(test_dates)

print(f'Feature count: {test_ft.shape[1]} columns')
print(f'\nDanh sách features:')
cols = [c for c in test_ft.columns if c != 'Date']
for i, c in enumerate(cols):
    print(f'  {i+1:3d}. {c}')

Feature count: 123 columns

Danh sách features:
    1. year
    2. month
    3. day
    4. dow
    5. doy
    6. dim
    7. quarter
    8. is_weekend
    9. days_to_eom
   10. days_from_som
   11. is_last1
   12. is_first1
   13. is_last2
   14. is_first2
   15. is_last3
   16. is_first3
   17. t_days
   18. t_years
   19. regime_pre2019
   20. regime_2019
   21. regime_post2019
   22. fourier_sin_y1
   23. fourier_cos_y1
   24. fourier_sin_y2
   25. fourier_cos_y2
   26. fourier_sin_y3
   27. fourier_cos_y3
   28. fourier_sin_y4
   29. fourier_cos_y4
   30. fourier_sin_y5
   31. fourier_cos_y5
   32. fourier_sin_w1
   33. fourier_cos_w1
   34. fourier_sin_w2
   35. fourier_cos_w2
   36. fourier_sin_m1
   37. fourier_cos_m1
   38. fourier_sin_m2
   39. fourier_cos_m2
   40. hol_new_year_days_diff
   41. on_new_year
   42. days_since_hol_new_year
   43. days_until_hol_new_year
   44. hol_womens_day_days_diff
   45. on_womens_day
   46. days_since_hol_womens_day
   47. days_until_hol_wom

## 4. Kiểm tra các fixes quan trọng

In [ ]:
# =====================================================
# CHECK 1: Fourier annual phải có bậc 5
# =====================================================
fourier_annual = [c for c in cols if 'sin_y' in c or 'cos_y' in c]
print('Fourier annual features:', fourier_annual)
assert 'fourier_sin_y5' in test_ft.columns, 'BUG: thiếu Fourier bậc 5!'
print('✓ Fourier annual OK — có bậc 1→5')

# =====================================================
# CHECK 2: Fourier weekly/monthly phải có bậc 2
# =====================================================
fourier_wm = [c for c in cols if ('sin_w' in c or 'cos_w' in c or 'sin_m' in c or 'cos_m' in c)]
print('\nFourier weekly/monthly features:', fourier_wm)
assert 'fourier_sin_w2' in test_ft.columns, 'BUG: thiếu weekly bậc 2!'
assert 'fourier_sin_m2' in test_ft.columns, 'BUG: thiếu monthly bậc 2!'
print('✓ Fourier weekly/monthly OK — có bậc 1→2')

# =====================================================
# CHECK 3: Promo 2024 phải được cover
# =====================================================
test_2024 = build_features(pd.date_range('2024-01-01', '2024-12-31'))
promo_2024 = test_2024['active_promo_count'].sum()
print(f'\nPromo 2024 coverage: {promo_2024} promo-days (phải > 0)')
assert promo_2024 > 0, 'BUG: không có promo nào cover năm 2024!'
print('✓ Promo 2024 OK')

# =====================================================
# CHECK 4: q3_odd phải = 1 trong Q3 2023 (năm lẻ)
# =====================================================
test_q3_2023 = build_features(pd.date_range('2023-07-01', '2023-09-30'))
assert test_q3_2023['q3_odd'].all() == 1, 'BUG: q3_odd không hoạt động!'
print('✓ q3_odd OK — Q3/2023 = 1')

# =====================================================
# CHECK 5: Tết 2023 (22/01/2023)
# =====================================================
test_tet = build_features(pd.date_range('2023-01-20', '2023-02-20'))
tet_day  = test_tet[test_tet['Date'] == '2023-01-22']
assert tet_day['tet_on'].values[0] == 1, 'BUG: tet_on sai!'
assert tet_day['tet_in_7'].values[0] == 1, 'BUG: tet_in_7 sai!'
after_30 = test_tet[test_tet['Date'] == '2023-02-18']
assert after_30['tet_after_14_30'].values[0] == 1, 'BUG: tet_after_14_30 sai!'
print('✓ Tết features OK')

print('\n=== TẤT CẢ CHECKS PASSED ===')

Fourier annual features: ['fourier_sin_y1', 'fourier_cos_y1', 'fourier_sin_y2', 'fourier_cos_y2', 'fourier_sin_y3', 'fourier_cos_y3', 'fourier_sin_y4', 'fourier_cos_y4', 'fourier_sin_y5', 'fourier_cos_y5']
✓ Fourier annual OK — có bậc 1→5

Fourier weekly/monthly features: ['fourier_sin_w1', 'fourier_cos_w1', 'fourier_sin_w2', 'fourier_cos_w2', 'fourier_sin_m1', 'fourier_cos_m1', 'fourier_sin_m2', 'fourier_cos_m2']
✓ Fourier weekly/monthly OK — có bậc 1→2

Promo 2024 coverage: 140 promo-days (phải > 0)
✓ Promo 2024 OK
✓ q3_odd OK — Q3/2023 = 1
✓ Tết features OK

=== TẤT CẢ CHECKS PASSED ===


## 5. Build train + test features và lưu ra file

In [ ]:
# Load data
sales      = pd.read_csv(f'{DATA_PATH}/sales.csv',             parse_dates=['Date'])
sample_sub = pd.read_csv(f'{DATA_PATH}/sample_submission.csv', parse_dates=['Date'])

sales      = sales.sort_values('Date').reset_index(drop=True)
sample_sub = sample_sub.sort_values('Date').reset_index(drop=True)

# Build features
print('Building train features...')
train_feat = build_features(sales['Date'])
train_feat = train_feat.merge(sales[['Date', 'Revenue', 'COGS']], on='Date', how='left')

print('Building test features...')
test_feat  = build_features(sample_sub['Date'])

# Feature columns
FEATURE_COLS = [c for c in train_feat.columns if c not in ['Date', 'Revenue', 'COGS']]

print(f'\nTrain: {train_feat.shape}')
print(f'Test : {test_feat.shape}')
print(f'Feature count: {len(FEATURE_COLS)}')

# Kiểm tra NaN
nan_counts = train_feat[FEATURE_COLS].isna().sum()
nan_cols   = nan_counts[nan_counts > 0]
if len(nan_cols) > 0:
    print(f'\nCác cột có NaN (bình thường cho window features):')
    for col, cnt in nan_cols.items():
        print(f'  {col}: {cnt} NaN ({cnt/len(train_feat)*100:.1f}%)')
else:
    print('\nKhông có NaN.')

Building train features...
Building test features...

Train: (3833, 125)
Test : (548, 123)
Feature count: 122

Các cột có NaN (bình thường cho window features):
  days_since_hol_new_year: 3763 NaN (98.2%)
  days_until_hol_new_year: 3756 NaN (98.0%)
  days_since_hol_womens_day: 3763 NaN (98.2%)
  days_until_hol_womens_day: 3763 NaN (98.2%)
  days_since_hol_reunification: 3763 NaN (98.2%)
  days_until_hol_reunification: 3763 NaN (98.2%)
  days_since_hol_labor_day: 3763 NaN (98.2%)
  days_until_hol_labor_day: 3763 NaN (98.2%)
  days_since_hol_national_day: 3756 NaN (98.0%)
  days_until_hol_national_day: 3756 NaN (98.0%)
  days_since_hol_vn_womens_day: 3756 NaN (98.0%)
  days_until_hol_vn_womens_day: 3756 NaN (98.0%)
  days_since_hol_dd_1111: 3756 NaN (98.0%)
  days_until_hol_dd_1111: 3756 NaN (98.0%)
  days_since_hol_dd_1212: 3756 NaN (98.0%)
  days_until_hol_dd_1212: 3756 NaN (98.0%)
  days_since_hol_christmas_eve: 3756 NaN (98.0%)
  days_until_hol_christmas_eve: 3756 NaN (98.0%)
  days_

In [ ]:
# Lưu features ra file để dùng trong training notebook
train_feat.to_parquet(f'{DATA_PATH}/train_features.parquet', index=False)
test_feat.to_parquet(f'{DATA_PATH}/test_features.parquet',   index=False)

# Lưu danh sách feature cols
import json
with open(f'{DATA_PATH}/feature_cols.json', 'w') as f:
    json.dump(FEATURE_COLS, f)

print('Đã lưu:')
print(f'  {DATA_PATH}/train_features.parquet')
print(f'  {DATA_PATH}/test_features.parquet')
print(f'  {DATA_PATH}/feature_cols.json')
print(f'  Total features: {len(FEATURE_COLS)}')

Đã lưu:
  /content/drive/MyDrive/AIO2026_Prj/Datathon2026/datathon-2026-round-1/train_features.parquet
  /content/drive/MyDrive/AIO2026_Prj/Datathon2026/datathon-2026-round-1/test_features.parquet
  /content/drive/MyDrive/AIO2026_Prj/Datathon2026/datathon-2026-round-1/feature_cols.json
  Total features: 122


In [ ]:
import os
# Liệt kê các tệp trong thư mục để xác nhận chúng đã được tạo vật lý
if os.path.exists(DATA_PATH):
    print(f"Danh sách tệp trong {DATA_PATH}:")
    print(os.listdir(DATA_PATH))
else:
    print("Thư mục không tồn tại. Vui lòng kiểm tra lại DATA_PATH.")

Danh sách tệp trong /content/drive/MyDrive/AIO2026_Prj/Datathon2026/datathon-2026-round-1:
['inventory.csv', 'baseline.ipynb', 'geography.csv', 'order_items.csv', 'customers.csv', 'payments.csv', 'orders.csv', 'shipments.csv', 'sales.csv', 'returns.csv', 'web_traffic.csv', 'sample_submission.csv', 'products.csv', 'reviews.csv', 'promotions.csv', 'train_features.parquet', 'test_features.parquet', 'feature_cols.json']


## 6. Cách load trong training notebook

```python
import pandas as pd
import json

DATA_PATH = '/content/drive/MyDrive/AIO2026_Prj/Datathon2026/datathon-2026-round-1'

train_feat   = pd.read_parquet(f'{DATA_PATH}/train_features.parquet')
test_feat    = pd.read_parquet(f'{DATA_PATH}/test_features.parquet')

with open(f'{DATA_PATH}/feature_cols.json') as f:
    FEATURE_COLS = json.load(f)

X_train = train_feat[FEATURE_COLS].values
y_rev   = np.log(train_feat['Revenue'].values)
y_cogs  = np.log(train_feat['COGS'].values)
X_test  = test_feat[FEATURE_COLS].values
```

**Lưu ý NaN trong window features:**  
Các cột `days_since_*`, `days_until_*`, `days_to_black_friday` có NaN ngoài window.  
LightGBM xử lý NaN tự nhiên — không cần fillna.  
Ridge cần fillna trước khi StandardScaler:
```python
X_train_ridge = train_feat[FEATURE_COLS].fillna(0).values
X_test_ridge  = test_feat[FEATURE_COLS].fillna(0).values
```

## 7. Tóm tắt tất cả fixes và additions

In [ ]:
# So sánh feature count với teammate
teammate_count = (
    10  # calendar basics
    + 6   # edge-of-month
    + 5   # trend/regime
    + 8   # Fourier annual bậc 1-4 (bug: chỉ có 4 bậc)
    + 2   # Fourier weekly bậc 1
    + 2   # Fourier monthly bậc 1
    + 40  # holiday distance (10 × 4)
    + 7   # tet features
    + 1   # black friday
    + 24  # promo (6 × 4)
    + 1   # is_odd_year
)

optimized_count = len(FEATURE_COLS)

print('=== So sánh feature count ===')
print(f'Teammate (gốc): ~{teammate_count} features')
print(f'Optimized     : {optimized_count} features')
print(f'Tăng thêm     : {optimized_count - teammate_count} features')

print('\n=== Features mới thêm ===')
new_features = [
    ('fourier_sin/cos_y5',        'Fourier annual bậc 5 (fix bug range)'),
    ('fourier_sin/cos_w2',        'Fourier weekly bậc 2 (thêm mới)'),
    ('fourier_sin/cos_m2',        'Fourier monthly bậc 2 (thêm mới)'),
    ('tet_before_14',             'Window 14 ngày trước Tết'),
    ('tet_after_14_30',           'Spike 14-30 ngày sau Tết'),
    ('days_since_tet',            'Continuous window sau Tết (từ teammate)'),
    ('days_until_tet',            'Continuous window trước Tết (từ teammate)'),
    ('q3_odd',                    'Q3 × năm lẻ — urban_blowout interaction'),
    ('q3_even',                   'Q3 × năm chẵn'),
    ('black_friday_window',       'Black Friday window ±3 ngày'),
    ('days_to_black_friday',      'Countdown đến Black Friday'),
    ('active_promo_count',        'Tổng số promo đang active'),
    ('promo_any_active',          'Có bất kỳ promo nào không'),
    ('promo_stackable',           'Có 2+ promo cùng lúc'),
]
for col, desc in new_features:
    print(f'  + {col:<30} {desc}')

=== So sánh feature count ===
Teammate (gốc): ~106 features
Optimized     : 122 features
Tăng thêm     : 16 features

=== Features mới thêm ===
  + fourier_sin/cos_y5             Fourier annual bậc 5 (fix bug range)
  + fourier_sin/cos_w2             Fourier weekly bậc 2 (thêm mới)
  + fourier_sin/cos_m2             Fourier monthly bậc 2 (thêm mới)
  + tet_before_14                  Window 14 ngày trước Tết
  + tet_after_14_30                Spike 14-30 ngày sau Tết
  + days_since_tet                 Continuous window sau Tết (từ teammate)
  + days_until_tet                 Continuous window trước Tết (từ teammate)
  + q3_odd                         Q3 × năm lẻ — urban_blowout interaction
  + q3_even                        Q3 × năm chẵn
  + black_friday_window            Black Friday window ±3 ngày
  + days_to_black_friday           Countdown đến Black Friday
  + active_promo_count             Tổng số promo đang active
  + promo_any_active               Có bất kỳ promo nào không
  + pr